# LeWM Duckietown Training Only

This notebook runs **training only** (no probe/eval) using the LeWM Hydra trainer in `/content/le-wm`.


In [ ]:
# 1) Training configuration
RUN_NAME = 'duckie_explore_retrain'
DATA_LOCAL = '/content/duckie_explore.h5'
MAX_EPOCHS = 50
BATCH_SIZE = 128
NUM_WORKERS = 2
PRECISION = 'bf16-mixed'

# Optional: set this to a checkpoint if you want resume/init
INIT_CKPT = ''


In [ ]:
# 2) Environment checks
import os, subprocess
print('python:', subprocess.check_output(['python3','--version'], text=True).strip())
print('data exists:', os.path.exists(DATA_LOCAL), DATA_LOCAL)
if not os.path.exists('/content/le-wm/train.py'):
    raise FileNotFoundError('Missing /content/le-wm/train.py. Clone/setup le-wm first.')


In [ ]:
# 3) Launch training (writes log to /content/train_duckie.log)
import subprocess

overrides = [
    f'data=duckietown',
    f'subdir={RUN_NAME}',
    f'data.path={DATA_LOCAL}',
    f'trainer.max_epochs={MAX_EPOCHS}',
    f'loader.batch_size={BATCH_SIZE}',
    f'loader.num_workers={NUM_WORKERS}',
    f'trainer.precision={PRECISION}',
    'wandb.enabled=false',
]
if INIT_CKPT:
    overrides.append(f'checkpoint.path={INIT_CKPT}')

cmd = 'cd /content/le-wm && HYDRA_FULL_ERROR=1 python3 train.py ' + ' '.join(overrides) + ' 2>&1 | tee /content/train_duckie.log'
print(cmd)
ret = subprocess.run(['bash','-lc',cmd])
if ret.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {ret.returncode}. See /content/train_duckie.log')
print('Training finished.')


In [ ]:
# 4) Locate checkpoints
import glob, os
cands = sorted(glob.glob(f'/content/stable-wm/{RUN_NAME}/*.ckpt'))
if not cands:
    print('No checkpoints found under', f'/content/stable-wm/{RUN_NAME}')
else:
    for p in cands[-10:]:
        print(p, os.path.getsize(p))


In [ ]:
# 5) Optional live monitor (run in separate cell while training is active)
# !tail -f /content/train_duckie.log
# !watch -n 30 'ls -lh /content/stable-wm/duckie_explore_retrain/*.ckpt 2>/dev/null || echo no-ckpt-yet'
